In [ ]:
# Cell 1: imports, parameters, and data loading (fast)
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.linalg import expm
from scipy.optimize import minimize
from scipy.signal import correlate
import warnings
warnings.filterwarnings('ignore')

# filterpy for UKF sigma points
try:
    from filterpy.kalman import MerweScaledSigmaPoints
except Exception as e:
    raise ImportError("Install filterpy: pip install filterpy")

# Circuit parameters
C1 = 30.14e-6
C2 = 185.6e-6
L  = 52.28
R  = 1673.0
R_L = 0.0
Ga = -0.801e-3
Gb = -0.365e-3
E  = 1.74

Ts = 0.01  # sampling time

dados_file = 'pcchua_dados.dat'
pert_file  = 'pcchua_pert.dat'

def load_pcchua(fname):
    data = np.loadtxt(fname, comments='#')
    t  = data[:,0]
    x  = data[:,1]
    y  = data[:,2]
    z  = data[:,3]
    ux = data[:,7]
    uy = data[:,8]
    uz = data[:,9]
    return {'t':t, 'x':x, 'y':y, 'z':z, 'ux':ux, 'uy':uy, 'uz':uz}

dados = load_pcchua(dados_file)
pert  = load_pcchua(pert_file)

t = dados['t']
y_vC2 = dados['y']
y_iL  = dados['z']
Y = np.vstack([y_vC2, y_iL]).T
N = len(t)
assert np.allclose(t[1]-t[0], Ts, atol=1e-2), "Timebase mismatch with Ts"


In [ ]:
# Cell 2: piecewise G, continuous dynamics, analytic continuous Jacobian A(x), and fast RK4 step
r_x_interp = interp1d(pert['t'], pert['ux'], bounds_error=False)
r_y_interp = interp1d(pert['t'], pert['uy'], bounds_error=False)
r_z_interp = interp1d(pert['t'], pert['uz'], bounds_error=False)

def G_of_v(v):
    av = abs(v)
    if av < E:
        return Ga
    return Gb + (Ga - Gb) * E / (av + 1e-12)

def chua_continuous_rhs_state(t_abs, state):
    v1, v2, iL = state
    rx = float(r_x_interp(t_abs))
    ry = float(r_y_interp(t_abs))
    rz = float(r_z_interp(t_abs))
    Gv = G_of_v(v1)
    dv1 = ( (v2 - v1)/R - Gv * v1 + rx ) / C1
    dv2 = ( (v1 - v2)/R + iL + ry ) / C2
    di  = ( -v2 + R_L * iL + rz ) / L
    return np.array([dv1, dv2, di])

def A_continuous(state):
    """
    Analytic continuous-time Jacobian A = df/dx evaluated at given state.
    dG/dv approximated as zero except at threshold; piecewise model uses dG/dv=0.
    """
    v1, v2, iL = state
    Gv = G_of_v(v1)
    # partials
    a11 = ( -1.0/R - Gv ) / C1
    a12 = ( 1.0 / R ) / C1
    a13 = 0.0
    a21 = ( 1.0 / R ) / C2
    a22 = ( -1.0 / R ) / C2
    a23 = 1.0 / C2
    a31 = 0.0
    a32 = -1.0 / L
    a33 = R_L / L
    A = np.array([[a11, a12, a13],
                  [a21, a22, a23],
                  [a31, a32, a33]])
    return A

def rk4_step(state, t_abs, dt):
    """
    Fixed-step RK4 integrator using chua_continuous_rhs_state with time-varying inputs.
    Fast and deterministic.
    """
    k1 = chua_continuous_rhs_state(t_abs, state)
    k2 = chua_continuous_rhs_state(t_abs + dt/2.0, state + dt*k1/2.0)
    k3 = chua_continuous_rhs_state(t_abs + dt/2.0, state + dt*k2/2.0)
    k4 = chua_continuous_rhs_state(t_abs + dt, state + dt*k3)
    return state + dt*(k1 + 2.0*k2 + 2.0*k3 + k4)/6.0


In [ ]:
# Cell 3: fast discretization f_d(x) and discretized Jacobian F via expm(A*Ts)
def discrete_propagation(xk, tk, Ts):
    """Return x_{k+1} via a single RK4 step from tk -> tk+Ts."""
    return rk4_step(xk, tk, Ts)

def discrete_F_via_expm(xk, tk, Ts):
    """
    Compute discrete-time Jacobian F = expm(A(xk) * Ts).
    Using continuous A(xk) (approx constant over the interval).
    This is cheap and more accurate than first-order Taylor.
    """
    A = A_continuous(xk)
    F = expm(A * Ts)
    return F


In [ ]:
# Cell 4: measurement model and EKF implementation using analytic F
def h_of_x(x):
    return np.array([x[1], x[2]])

def H_matrix(x):
    return np.array([[0.0, 1.0, 0.0],
                     [0.0, 0.0, 1.0]])

def run_ekf_fast(Y, t, x0, P0, Q, R, Ts, subset_for_tune=None):
    N = Y.shape[0]
    x_est = np.zeros((N, 3))
    P_est = np.zeros((N, 3, 3))
    innov  = np.zeros((N, 2))

    x = x0.copy()
    P = P0.copy()

    for k in range(N):
        tk = t[k]
        # Predict
        x_pred = discrete_propagation(x, tk, Ts)
        Fk = discrete_F_via_expm(x, tk, Ts)   # analytic discrete Jacobian
        P_pred = Fk @ P @ Fk.T + Q

        # Update
        Hk = H_matrix(x_pred)
        z_pred = h_of_x(x_pred)
        z = Y[k]
        y_tilde = z - z_pred
        S = Hk @ P_pred @ Hk.T + R
        K = P_pred @ Hk.T @ np.linalg.inv(S)

        x = x_pred + K @ y_tilde
        P = (np.eye(3) - K @ Hk) @ P_pred

        x_est[k] = x
        P_est[k] = P
        innov[k] = y_tilde

    return x_est, P_est, innov


In [ ]:
# Cell 5: Fast UKF implementation (manual sigma-point UKF) using RK4 propagation
def ukf_manual(Y, t, x0, P0, Q, R, Ts, alpha=0.1, beta=2.0, kappa=0.0):
    n = 3
    m = 2
    # prepare Merwe weights and sigma points
    points = MerweScaledSigmaPoints(n, alpha=alpha, beta=beta, kappa=kappa)
    Wm = points.Wm
    Wc = points.Wc

    x = x0.copy()
    P = P0.copy()

    N = Y.shape[0]
    x_est = np.zeros((N, n))
    P_est = np.zeros((N, n, n))
    innov  = np.zeros((N, m))

    for k in range(N):
        tk = t[k]
        # Generate sigma points (2n+1, n)
        sigmas = points.sigma_points(x, P)
        # Propagate sigma points through dynamics (each with same tk)
        sigmas_pred = np.array([discrete_propagation(s, tk, Ts) for s in sigmas])
        # Predicted mean and covariance
        x_pred = np.sum(Wm[:,None] * sigmas_pred, axis=0)
        P_pred = Q.copy()
        for i in range(sigmas_pred.shape[0]):
            d = sigmas_pred[i] - x_pred
            P_pred += Wc[i] * np.outer(d, d)

        # Predicted measurements from sigma points
        z_sig = np.array([h_of_x(s) for s in sigmas_pred])
        z_pred = np.sum(Wm[:,None] * z_sig, axis=0)
        P_zz = R.copy()
        P_xz = np.zeros((n, m))
        for i in range(sigmas_pred.shape[0]):
            dz = z_sig[i] - z_pred
            dx = sigmas_pred[i] - x_pred
            P_zz += Wc[i] * np.outer(dz, dz)
            P_xz += Wc[i] * np.outer(dx, dz)

        # Kalman gain and update
        K = P_xz @ np.linalg.inv(P_zz)
        z = Y[k]
        y_tilde = z - z_pred
        x = x_pred + K @ y_tilde
        P = P_pred - K @ P_zz @ K.T

        x_est[k] = x
        P_est[k] = P
        innov[k] = y_tilde

    return x_est, P_est, innov


In [ ]:
# Cell 6: whiteness metric and fast tuning via Nelder-Mead on a short subset
def whiteness_metric(innov, maxlag=10):
    N, m = innov.shape
    metric = 0.0
    for j in range(m):
        s = innov[:, j] - np.mean(innov[:, j])
        var = np.var(s)
        if var < 1e-12:
            continue
        for lag in range(1, maxlag+1):
            r = np.correlate(s[:-lag], s[lag:])[0] / (N - lag)
            metric += abs(r / var)
    return metric / m

# Base Q and R choices (tunable)
Q_base = np.diag([1e-6, 1e-6, 1e-7])
R_base = np.diag([ (np.std(y_vC2[:200])**2 + 1e-8), (np.std(y_iL[:200])**2 + 1e-8) ])

# small subset for tuning to reduce runtime
tune_len = min(2000, N)  # use first up to 2000 samples
Y_tune = Y[:tune_len]
t_tune = t[:tune_len]
x0 = np.array([0.0, y_vC2[0], y_iL[0]])
P0 = np.diag([1.0, 0.1, 0.1])

def objective_scalars(s):
    # s = [log_q_scale, log_r_scale] optimize in log-space to keep positivity
    q_scale = 10**s[0]
    r_scale = 10**s[1]
    Q_try = Q_base * q_scale
    R_try = R_base * r_scale
    # run a fast filter (EKF here) on short subset
    x_est, P_est, innov = run_ekf_fast(Y_tune, t_tune, x0, P0, Q_try, R_try, Ts)
    return whiteness_metric(innov, maxlag=8)

# initial guess near 0 (q_scale=1, r_scale=1 -> logs = 0)
res = minimize(objective_scalars, x0=np.array([0.0, 0.0]), method='Nelder-Mead',
               options={'maxiter':60, 'xatol':1e-2, 'fatol':1e-3})
q_scale_opt = 10**res.x[0]
r_scale_opt = 10**res.x[1]
print("Tuning finished:", res.message)
print("Found q_scale =", q_scale_opt, " r_scale =", r_scale_opt)


In [ ]:
# Cell 7: final EKF and UKF runs on full dataset with tuned scalars, plots & diagnostics
Q_tuned = Q_base * q_scale_opt
R_tuned = R_base * r_scale_opt

# EKF final
x_ekf, P_ekf, innov_ekf = run_ekf_fast(Y, t, x0, P0, Q_tuned, R_tuned, Ts)

# UKF final (manual)
x_ukf, P_ukf, innov_ukf = ukf_manual(Y, t, x0, P0, Q_tuned, R_tuned, Ts)

# Plot estimated vC1 from both filters
plt.figure(figsize=(10,6))
plt.plot(t, x_ekf[:,0], label='vC1 EKF (est)', lw=1)
plt.plot(t, x_ukf[:,0], label='vC1 UKF (est)', lw=1, alpha=0.9)
plt.xlabel('time (s)')
plt.ylabel('vC1 (V)')
plt.legend()
plt.grid(True)
plt.title('Estimated vC1 (EKF vs UKF)')
plt.tight_layout()
plt.show()

# Plot innovations and ACFs quickly for both filters (shorter lags)
def plot_innov_and_acf_quick(innov, title):
    fig, axs = plt.subplots(2,2,figsize=(10,5))
    axs = axs.ravel()
    axs[0].plot(t, innov[:,0]); axs[0].set_title(title + ' innov vC2'); axs[0].grid(True)
    axs[1].plot(t, innov[:,1]); axs[1].set_title(title + ' innov iL'); axs[1].grid(True)
    for i in range(2):
        s = innov[:,i] - np.mean(innov[:,i])
        var_s = np.var(s)
        if var_s < 1e-12:
            var_s = 1.0
        acf_full = np.real(correlate(s, s, mode='full')) / var_s / len(s)
        mid = len(acf_full)//2
        lags = np.arange(-50,51)
        acf_plot = acf_full[mid-50:mid+51]
        axs[2+i].stem(lags, acf_plot, use_line_collection=True)
        axs[2+i].set_title(title + f' ACF ch {i}'); axs[2+i].grid(True)
    plt.tight_layout()
    plt.show()

plot_innov_and_acf_quick(innov_ekf, 'EKF (tuned)')
plot_innov_and_acf_quick(innov_ukf, 'UKF (tuned)')

print("Whiteness metric EKF:", whiteness_metric(innov_ekf, maxlag=10))
print("Whiteness metric UKF:", whiteness_metric(innov_ukf, maxlag=10))
